# Bank Customer Churn Prediction — ANN with SMOTE 

**Dataset:** Churn_Modelling.csv (10,000 bank customers, 20.4% churn rate)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, roc_curve, auc, accuracy_score,
                              precision_score, recall_score, f1_score,
                              ConfusionMatrixDisplay, classification_report)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
%matplotlib inline


## 1. Load raw data

In [ ]:
df = pd.read_csv("../data/Churn_Modelling.csv")
print("Raw shape:", df.shape)
df.head()


In [ ]:
df.info()


## 2. Dataset analysis — class balance, missing values

Required by CO3 (dataset distribution / class imbalance analysis).

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nClass distribution (Exited):")
print(df["Exited"].value_counts())
print(df["Exited"].value_counts(normalize=True).round(4))

sns.countplot(x="Exited", data=df)
plt.title("Class Distribution — Exited (0 = Stayed, 1 = Churned)")
plt.show()


## 3. Data Cleaning & Preprocessing

Full step-by-step cleaning shown here (this is the same logic packaged into
`src/preprocessing.py` for the automated pipeline, reproduced inline here for
transparency):

1. Drop identifier columns with no predictive value (RowNumber, CustomerId, Surname)
2. Impute missing numeric values with the median, categorical with the mode (defensive —
   this dataset has zero missing values, verified above)
3. Never impute the target column; drop any row with a missing label instead
4. Label-encode binary categorical columns (Gender)
5. One-hot encode multi-class categorical columns (Geography), dropping the first level
   to avoid the dummy-variable trap


In [ ]:
TARGET_COL = "Exited"
ID_COLS_TO_DROP = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
df_clean = df.drop(columns=ID_COLS_TO_DROP)
print("Dropped ID columns:", ID_COLS_TO_DROP)

candidate_cols = [c for c in df_clean.columns if c != TARGET_COL]
numeric_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(df_clean[c])]
categorical_cols = [c for c in candidate_cols if not pd.api.types.is_numeric_dtype(df_clean[c])]
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# Missing-value handling (defensive; this dataset has none)
for c in numeric_cols:
    if df_clean[c].isnull().sum() > 0:
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())
for c in categorical_cols:
    if df_clean[c].isnull().sum() > 0:
        df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])
if df_clean[TARGET_COL].isnull().sum() > 0:
    df_clean = df_clean.dropna(subset=[TARGET_COL])

# Encode categoricals: binary -> label encode, multi-class -> one-hot (drop_first)
for c in categorical_cols:
    if df_clean[c].nunique() <= 2:
        df_clean[c] = LabelEncoder().fit_transform(df_clean[c])
        print(f"Label-encoded '{c}' (binary)")
    else:
        dummies = pd.get_dummies(df_clean[c], prefix=c, drop_first=True).astype(int)
        df_clean = pd.concat([df_clean.drop(columns=[c]), dummies], axis=1)
        print(f"One-hot encoded '{c}' -> {list(dummies.columns)}")

feature_cols = [c for c in df_clean.columns if c != TARGET_COL]
print("\nFinal feature columns (", len(feature_cols), "):", feature_cols)
df_clean.head()


## 4. Train/Test split + feature scaling (no leakage)

In [ ]:
X = df_clean[feature_cols].values
y = df_clean[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Fit scaler on TRAIN ONLY, then apply to both train and test (prevents data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
n_features = X_train.shape[1]

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class balance:", np.bincount(y_train) / len(y_train))
print("Test class balance :", np.bincount(y_test) / len(y_test))


In [ ]:
def get_class_weight(y):
    n_pos, n_neg = y.sum(), len(y) - y.sum()
    return {0: len(y) / (2 * n_neg), 1: len(y) / (2 * n_pos)}

class_weight = get_class_weight(y_train)
print("Class weights (Baseline model):", class_weight)


## 5. ANN Architecture

Full Keras model definition shown here layer by layer:

- **Input layer:** shape = (n_features,) = 11 engineered features
- **Hidden layer 1:** Dense(16 units, ReLU activation)
- **Hidden layer 2:** Dense(8 units, ReLU activation)
- **Output layer:** Dense(1 unit, Sigmoid activation) — binary churn probability
- **Optimizer:** Adam, learning rate = 0.001
- **Loss:** Binary Crossentropy


In [ ]:
def build_ann(input_dim, regularizer=None, l_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(16, activation="relu", kernel_regularizer=regularizer),
        layers.Dense(8, activation="relu", kernel_regularizer=regularizer),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=l_rate),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

sample_model = build_ann(n_features)
sample_model.summary()


## 6. Train Baseline ANN (class_weight only)

In [ ]:
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

baseline_model = build_ann(n_features)
baseline_history = baseline_model.fit(
    X_train, y_train, validation_split=0.2, epochs=150, batch_size=32,
    class_weight=class_weight, callbacks=[early_stop], verbose=1
)
print(f"Baseline trained for {len(baseline_history.history['loss'])} epochs (early stopping).")


## 6b. 5-Fold Stratified Cross-Validation (k = 5) — Baseline ANN

Checks that the baseline's performance is stable across different train/validation partitions and is not an artefact of one lucky split. Same architecture, class_weight handling and early stopping as the baseline; each fold trains a fresh model.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    np.random.seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)
    m = build_ann(n_features)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    m.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=100, batch_size=32,
          class_weight=get_class_weight(y_tr), callbacks=[es], verbose=0)

    proba = m.predict(X_va, verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)
    cv_rows.append(dict(fold=fold,
                        accuracy=accuracy_score(y_va, pred),
                        precision=precision_score(y_va, pred),
                        recall=recall_score(y_va, pred),
                        f1=f1_score(y_va, pred),
                        roc_auc=roc_auc_score(y_va, proba)))
    print(f"Fold {fold}: " + "  ".join(f"{k}={v:.4f}" for k, v in cv_rows[-1].items() if k != "fold"))

cv_df = pd.DataFrame(cv_rows).set_index("fold")
summary = pd.DataFrame({"mean": cv_df.mean(), "std": cv_df.std(ddof=0)}).round(4)
print(f"\n{K}-fold CV summary (mean / std):")
print(summary)

plt.figure(figsize=(6.5, 4.8))
plt.bar(cv_df.index, cv_df["roc_auc"], color="#2563eb", alpha=0.85)
plt.axhline(cv_df["roc_auc"].mean(), color="red", linestyle="--",
            label=f"Mean AUC = {cv_df['roc_auc'].mean():.3f}")
plt.xlabel("Fold"); plt.ylabel("ROC-AUC")
plt.title(f"{K}-Fold Stratified CV — ROC-AUC per Fold (Baseline ANN)")
plt.legend(); plt.tight_layout(); plt.show()


## 7. CO5 Innovation — Train Proposed ANN with SMOTE

Instead of (or in addition to) reweighting the loss function, SMOTE synthesizes new minority-class ('Exited') samples in feature space so the network sees a genuinely balanced training distribution. Same architecture, same hyperparameters as the baseline — only the imbalance-handling strategy changes, which isolates it as the sole controlled variable.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"SMOTE resampled: {len(y_train)} -> {len(y_train_res)} samples")
print("New class balance:", np.bincount(y_train_res) / len(y_train_res))

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

proposed_model = build_ann(n_features)
early_stop2 = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
proposed_history = proposed_model.fit(
    X_train_res, y_train_res, validation_split=0.2, epochs=150, batch_size=32,
    callbacks=[early_stop2], verbose=1
)
print(f"Proposed trained for {len(proposed_history.history['loss'])} epochs (early stopping).")


## 8. Evaluate both models on the same held-out test set

In [ ]:
def evaluate_model(model, X_test, y_test, threshold=0.5, label="Model"):
    y_proba = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_proba >= threshold).astype(int)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    print(f"\n--- {label} Test Set Metrics ---")
    print(f"Accuracy   : {acc:.4f}\nPrecision  : {prec:.4f}\nRecall     : {rec:.4f}"
          f"\nF1-score   : {f1:.4f}\nSpecificity: {specificity:.4f}\nROC-AUC    : {roc_auc:.4f}\n")
    print(classification_report(y_test, y_pred, target_names=["Stayed", "Exited"]))
    return dict(cm=cm, fpr=fpr, tpr=tpr, roc_auc=roc_auc, accuracy=acc,
                precision=prec, recall=rec, f1=f1, specificity=specificity)

baseline_results = evaluate_model(baseline_model, X_test, y_test, label="Baseline ANN")
proposed_results = evaluate_model(proposed_model, X_test, y_test, label="Proposed ANN (SMOTE)")


## 9. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ConfusionMatrixDisplay(baseline_results["cm"], display_labels=["Stayed", "Exited"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
axes[0].set_title("Baseline ANN")
ConfusionMatrixDisplay(proposed_results["cm"], display_labels=["Stayed", "Exited"]).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format="d")
axes[1].set_title("Proposed ANN (SMOTE)")
plt.tight_layout(); plt.show()


## 10. ROC curve comparison

In [ ]:
plt.figure(figsize=(6.5, 6))
plt.plot(baseline_results["fpr"], baseline_results["tpr"],
         label=f"Baseline (AUC={baseline_results['roc_auc']:.3f})", color="#94a3b8", lw=2)
plt.plot(proposed_results["fpr"], proposed_results["tpr"],
         label=f"Proposed SMOTE (AUC={proposed_results['roc_auc']:.3f})", color="#2563eb", lw=2)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Baseline vs Proposed"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()


## 11. Training vs. validation curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].plot(baseline_history.history["loss"], label="Train Loss")
axes[0, 0].plot(baseline_history.history["val_loss"], label="Val Loss")
axes[0, 0].set_title("Baseline — Loss over Epochs"); axes[0, 0].legend()
axes[0, 1].plot(baseline_history.history["accuracy"], label="Train Accuracy")
axes[0, 1].plot(baseline_history.history["val_accuracy"], label="Val Accuracy")
axes[0, 1].set_title("Baseline — Accuracy over Epochs"); axes[0, 1].legend()
axes[1, 0].plot(proposed_history.history["loss"], label="Train Loss")
axes[1, 0].plot(proposed_history.history["val_loss"], label="Val Loss")
axes[1, 0].set_title("Proposed (SMOTE) — Loss over Epochs"); axes[1, 0].legend()
axes[1, 1].plot(proposed_history.history["accuracy"], label="Train Accuracy")
axes[1, 1].plot(proposed_history.history["val_accuracy"], label="Val Accuracy")
axes[1, 1].set_title("Proposed (SMOTE) — Accuracy over Epochs"); axes[1, 1].legend()
plt.tight_layout(); plt.show()


## 12. Ablation summary 

| Experiment | Imbalance Handling | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|---|
| Baseline | class_weight only | 0.7745 | 0.4669 | 0.7617 | 0.5789 | 0.8521 |
| Proposed | SMOTE oversampling | 0.8125 | 0.5303 | 0.6880 | 0.5989 | 0.8540 |

**Observation:** SMOTE improves accuracy (+3.8 pts), precision (+6.3 pts), F1 (+2.0 pts), and AUC (+0.19 pts)
over the class-weighted baseline, at the cost of a modest drop in recall (-7.4 pts). This is an expected and
explainable trade-off: class_weight only reweights the loss gradient, so the model still trains on the
same 20.4% real minority examples and tends to over-predict the minority class to compensate, inflating recall
at the expense of precision. SMOTE instead gives the network new synthetic minority examples to actually learn
decision-boundary structure from, producing a more balanced, higher-precision.


**Note on reproducibility:** metric values in the executed outputs above (including the 5-fold CV) may differ by a small margin (typically ~1-2 percentage points) from those in README.md / results/*.csv. Both are genuine, independently executed runs of the identical architecture, hyperparameters and random_state=42; the variation is expected because TensorFlow training is not bit-for-bit deterministic across separate processes/environments even with a fixed seed. README.md and results/*.csv reflect the canonical pipeline run via src/train.py, src/test.py and src/cross_validation.py; this notebook is an independently reproducible walkthrough of the same method.